# Speech Emotion Recognition (SER) - Exploratory Data Analysis
This notebook provides a detailed visualization and analysis of the Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS) dataset used for Speech Emotion Recognition using Deep Learning.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

## 1. Load Dataset Metadata
Define the mappings for RAVDESS emotions.

In [ ]:
EMOTIONS = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

dataset_path = "../dataset/raw"
audio_files = glob.glob(os.path.join(dataset_path, "Actor_*", "*.wav"))
print(f"Total audio files found: {len(audio_files)}")

## 2. Visualize Waveform and Spectrogram for Different Emotions
Let's write a function to plot the audio waveform and its corresponding mel spectrogram side-by-side.

In [ ]:
def plot_audio_properties(file_path, emotion_name):
    # Load audio
    y, sr = librosa.load(file_path, res_type='kaiser_fast')
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
    
    # 1. Waveform
    librosa.display.waveshow(y, sr=sr, ax=ax1, color='purple')
    ax1.set_title(f'Waveform - {emotion_name.upper()}')
    ax1.set_xlabel('Time (seconds)')
    ax1.set_ylabel('Amplitude')
    
    # 2. Spectrogram
    stft_data = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    img = librosa.display.specshow(stft_data, sr=sr, x_axis='time', y_axis='hz', ax=ax2, cmap='magma')
    ax2.set_title(f'Spectrogram - {emotion_name.upper()}')
    fig.colorbar(img, ax=ax2, format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

Select and display one file for a few emotions (e.g., Happy, Angry, Sad).

In [ ]:
# Find a file representing happy (03), sad (04), angry (05)
sample_files = {}
for f in audio_files:
    name = os.path.basename(f)
    parts = name.split('-')
    if len(parts) >= 7:
        emotion = EMOTIONS[parts[2]]
        if emotion not in sample_files and emotion in ['happy', 'sad', 'angry']:
            sample_files[emotion] = f
            
for emotion, filepath in sample_files.items():
    plot_audio_properties(filepath, emotion)

## 3. Feature Distribution Analysis
Examine the MFCC, Chroma, and Mel spectrogram mean shapes.

In [ ]:
if os.path.exists("../dataset/extracted_features.csv"):
    df = pd.read_csv("../dataset/extracted_features.csv")
    
    # Plot distribution of emotions
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x='emotion', palette='viridis')
    plt.title('Distribution of Emotions in Dataset')
    plt.xlabel('Emotion')
    plt.ylabel('Count')
    plt.show()
else:
    print("Features CSV cache not found. Train the model first to generate cache file!")